In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import row_number, col, to_date, year, month, day, initcap, dense_rank, trim, sum, count, desc, avg, asc, countDistinct 
from pyspark.sql.window import Window

session = SparkSession.builder.appName("Megamart Exercise").master("local[*]").getOrCreate()

In [2]:
data = session.read.csv("Megamart.csv", header= True)

In [3]:
data.show(5)


+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+
|order_id|user_id|order_date|product_id|product_category|product_name|quantity|price_per_unit|payment_method|order_status|
+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+
|    1001|   U188|2025-04-20|      P940|         Fashion|    Sneakers|       2|         58.53|        PayPal|   Cancelled|
|    1002|   U062|2025-04-16|      P794|         Fashion|     T-Shirt|       3|         83.76|           UPI|    Returned|
|    1003|   U058|2025-04-18|      P326|         Fashion|  Sunglasses|       2|         78.85|        PayPal|  Processing|
|    1004|   U011|2025-04-10|      P574|         Fashion|  Sunglasses|       5|         46.49|        PayPal|   Delivered|
|    1005|   U003|2025-04-19|      P988|      Home Decor| Photo Frame|       2|         78.61|        PayPal|    Returned|
+--------+------

In [4]:
#Convert order_date from string to a DateType and extract year, month, day.

data = data.withColumn("order_date", to_date("order_date"))
data_year = data.withColumn("year", year("order_date"))
data_month = data_year.withColumn("month", month("order_date"))
data_day = data_month.withColumn("day", day("order_date"))

data_day.select("order_date", 'year', 'month', "day").show(7)

data_day.printSchema()

+----------+----+-----+---+
|order_date|year|month|day|
+----------+----+-----+---+
|2025-04-20|2025|    4| 20|
|2025-04-16|2025|    4| 16|
|2025-04-18|2025|    4| 18|
|2025-04-10|2025|    4| 10|
|2025-04-19|2025|    4| 19|
|2025-04-15|2025|    4| 15|
|2025-04-23|2025|    4| 23|
+----------+----+-----+---+
only showing top 7 rows
root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price_per_unit: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)



In [5]:
#Create a new column total_price = quantity * price_per_unit.

data = data.withColumn("quantity", col("quantity").cast("int")).withColumn("price_per_unit", col("price_per_unit").cast("float"))
data.printSchema()

data = data.withColumn("total_price", col("quantity")*col("price_per_unit"))
data.select("order_id", "quantity", "price_per_unit", "total_price").show(10)

root
 |-- order_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price_per_unit: float (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)

+--------+--------+--------------+------------------+
|order_id|quantity|price_per_unit|       total_price|
+--------+--------+--------------+------------------+
|    1001|       2|         58.53|117.05999755859375|
|    1002|       3|         83.76| 251.2800064086914|
|    1003|       2|         78.85| 157.6999969482422|
|    1004|       5|         46.49|232.45000839233398|
|    1005|       2|         78.61|157.22000122070312|
|    1006|       4|         53.51| 214.0399932861328|
|    1007|       5|         12.71| 63.55000019073486|
|    1008|       1|      

In [6]:
#Standardize categorical values (e.g., ensure order_status has consistent formatting like "Delivered" vs "delivered").

data = data.withColumn("order_status", initcap(trim(col("order_status"))))

data.show(5)

+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+------------------+
|order_id|user_id|order_date|product_id|product_category|product_name|quantity|price_per_unit|payment_method|order_status|       total_price|
+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+------------------+
|    1001|   U188|2025-04-20|      P940|         Fashion|    Sneakers|       2|         58.53|        PayPal|   Cancelled|117.05999755859375|
|    1002|   U062|2025-04-16|      P794|         Fashion|     T-Shirt|       3|         83.76|           UPI|    Returned| 251.2800064086914|
|    1003|   U058|2025-04-18|      P326|         Fashion|  Sunglasses|       2|         78.85|        PayPal|  Processing| 157.6999969482422|
|    1004|   U011|2025-04-10|      P574|         Fashion|  Sunglasses|       5|         46.49|        PayPal|   Delivered|232.45000839233398|
|    1

In [7]:
#Filter all orders where order_status = 'Delivered'.

data.filter(col("order_status") == 'Delivered').show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|       total_price|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+
|    1004|   U011|2025-04-10|      P574|         Fashion|          Sunglasses|       5|         46.49|        PayPal|   Delivered|232.45000839233398|
|    1017|   U021|2025-04-26|      P728|           Books|  Big Data Explained|       5|         13.42|        PayPal|   Delivered| 67.10000038146973|
|    1019|   U115|2025-05-03|      P973|         Kitchen|             Blender|       3|         35.68|    Debit Card|   Delivered|107.04000091552734|
|    1022|   U027|2025-04-28|      P269|           Books|       AI Revolution|       5|         24.5

In [8]:
#Replace nulls (if any in extended dataset) in payment_method with "Unknown".

data_null_values = data.fillna({'payment_method': 'Unknown'})
data_null_values.filter(col('payment_method') == 'Unknown').show()

#which means there are no null values in payment_method column

+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+-----------+
|order_id|user_id|order_date|product_id|product_category|product_name|quantity|price_per_unit|payment_method|order_status|total_price|
+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+-----------+
+--------+-------+----------+----------+----------------+------------+--------+--------------+--------------+------------+-----------+



In [9]:
#Find the total revenue generated per product_category.

data.groupBy("product_category").agg(sum("total_price").alias("Revenue")).show()

+----------------+------------------+
|product_category|           Revenue|
+----------------+------------------+
|         Kitchen|32161.610142707825|
|         Fashion|31437.230010986328|
|     Electronics|34848.490067481995|
|           Books| 31514.62001800537|
|      Home Decor| 35400.28000450134|
+----------------+------------------+



In [10]:
#Count the number of orders per payment method.

data.groupBy("payment_method").agg(count("order_id").alias("Order_Counts")).show()

+--------------+------------+
|payment_method|Order_Counts|
+--------------+------------+
|   Credit Card|         251|
|        PayPal|         261|
|    Debit Card|         224|
|           UPI|         264|
+--------------+------------+



In [11]:
#Find the top 5 products by sales (total_price).

data.groupBy("product_name").agg(sum("total_price").alias("sales")).orderBy(desc("sales")).limit(5).show()

+-------------+------------------+
| product_name|             sales|
+-------------+------------------+
|   Wall Clock| 8241.020010948181|
|      T-Shirt| 7904.819952011108|
|Cushion Cover|7834.5300035476685|
|   Smartwatch| 7750.909991264343|
|  USB-C Cable| 7586.400054931641|
+-------------+------------------+



In [12]:
#Calculate the average order value per user_id.

data.groupBy("user_id").agg(avg("total_price").alias("Average_order_value")).show()

+-------+-------------------+
|user_id|Average_order_value|
+-------+-------------------+
|   U185| 330.02500438690186|
|   U175|  126.3599967956543|
|   U014| 141.96666844685873|
|   U034|              134.5|
|   U200| 156.39400062561035|
|   U160|  135.6241668065389|
|   U180| 176.23908857865766|
|   U094|   205.212003326416|
|   U067| 132.10999658372668|
|   U164| 139.42499780654907|
|   U127| 252.05000162124634|
|   U092| 112.73666890462239|
|   U039| 132.74625301361084|
|   U176| 230.03500175476074|
|   U090| 181.19750261306763|
|   U053| 109.87750339508057|
|   U133| 113.74499893188477|
|   U068|  273.9750032424927|
|   U080|   84.7249984741211|
|   U082|  391.8599967956543|
+-------+-------------------+
only showing top 20 rows


In [13]:
data.filter(col("order_status") == 'Cancelled').groupBy("product_category").count().orderBy(desc("count")).show(1)

+----------------+-----+
|product_category|count|
+----------------+-----+
|     Electronics|   64|
+----------------+-----+
only showing top 1 row


In [14]:
#Group orders by order_status and payment_method to see patterns (e.g., which payment mode has most cancellations).

data.groupBy(['order_status', 'payment_method']).count().orderBy(asc("order_status")).show()

+------------+--------------+-----+
|order_status|payment_method|count|
+------------+--------------+-----+
|   Cancelled|    Debit Card|   49|
|   Cancelled|           UPI|   75|
|   Cancelled|   Credit Card|   67|
|   Cancelled|        PayPal|   73|
|   Delivered|    Debit Card|   49|
|   Delivered|   Credit Card|   65|
|   Delivered|           UPI|   61|
|   Delivered|        PayPal|   63|
|  Processing|           UPI|   67|
|  Processing|    Debit Card|   71|
|  Processing|   Credit Card|   53|
|  Processing|        PayPal|   65|
|    Returned|    Debit Card|   55|
|    Returned|   Credit Card|   66|
|    Returned|           UPI|   61|
|    Returned|        PayPal|   60|
+------------+--------------+-----+



In [15]:
#2. Find users who purchased products from multiple categories.

data.groupBy("user_id") \
  .agg(countDistinct("product_category").alias("category_count")) \
  .filter(col("category_count") > 1) \
  .show()

+-------+--------------+
|user_id|category_count|
+-------+--------------+
|   U185|             3|
|   U014|             4|
|   U034|             2|
|   U200|             3|
|   U160|             4|
|   U180|             4|
|   U094|             3|
|   U067|             3|
|   U164|             2|
|   U092|             4|
|   U127|             3|
|   U039|             4|
|   U176|             2|
|   U053|             2|
|   U090|             3|
|   U068|             2|
|   U133|             2|
|   U082|             2|
|   U107|             3|
|   U077|             2|
+-------+--------------+
only showing top 20 rows


In [16]:
#Identify repeat customers (user_id with more than one order).

data.groupBy("user_id").count().filter(col("count") > 1).show()

+-------+-----+
|user_id|count|
+-------+-----+
|   U185|    4|
|   U175|    3|
|   U014|    6|
|   U034|    4|
|   U200|    5|
|   U160|   12|
|   U180|   11|
|   U094|    5|
|   U067|    9|
|   U164|    4|
|   U127|    4|
|   U092|    6|
|   U039|    8|
|   U176|    4|
|   U090|    8|
|   U053|    4|
|   U133|    2|
|   U068|    2|
|   U080|    2|
|   U082|    2|
+-------+-----+
only showing top 20 rows


In [17]:
#Rank products within each product category by revenue.

window_spec = Window.partitionBy("product_category").orderBy(desc("total_price"))

data.withColumn("rank", dense_rank().over(window_spec)).show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+----+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|       total_price|rank|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+----+
|    1954|   U121|2025-04-09|      P681|           Books|  Big Data Explained|       5|         95.13|        PayPal|    Returned|475.64998626708984|   1|
|    1149|   U096|2025-04-05|      P684|           Books|   Spark for Dummies|       5|         93.19|    Debit Card|  Processing|465.95001220703125|   2|
|    1136|   U023|2025-04-26|      P470|           Books|   Spark for Dummies|       5|         87.21|        PayPal|   Delivered| 436.0499954223633|   3|
|    1750|   U105|2025-04-18|      P999|           Books|          Cle

In [18]:
#For each user, calculate the running total spent over time.

window_spec = Window.partitionBy("user_id").orderBy("order_date")

data.withColumn("running_total", sum("total_price").over(window_spec)).show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+------------------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|       total_price|     running_total|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+------------------+
|    1685|   U001|2025-04-08|      P675|     Electronics|Bluetooth Headphones|       2|         84.77|    Debit Card|    Returned| 169.5399932861328| 169.5399932861328|
|    1024|   U001|2025-04-09|      P960|           Books|   Spark for Dummies|       3|         56.91|        PayPal|    Returned|170.72999954223633|340.26999282836914|
|    1494|   U001|2025-04-11|      P735|         Fashion|            Sneakers|       3|         26.31|    Debit Card|  Processing| 78.92999839782715| 655.9

In [19]:
#Find the latest order per user using row number or rank.

window_spec = Window.partitionBy("user_id").orderBy(desc("order_date"))

data.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).show()

+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+-------+
|order_id|user_id|order_date|product_id|product_category|        product_name|quantity|price_per_unit|payment_method|order_status|       total_price|row_num|
+--------+-------+----------+----------+----------------+--------------------+--------+--------------+--------------+------------+------------------+-------+
|    1716|   U001|2025-04-27|      P390|         Fashion|          Sunglasses|       3|         29.09|   Credit Card|  Processing| 87.27000045776367|      1|
|    1392|   U002|2025-04-27|      P595|         Kitchen|             Blender|       5|         43.62|        PayPal|  Processing|218.09999465942383|      1|
|    1810|   U003|2025-05-01|      P599|         Kitchen|           Microwave|       5|         71.66|           UPI|   Cancelled| 358.3000183105469|      1|
|    1212|   U004|2025-05-02|      P302|         Kit

In [20]:
#Calculate daily sales trend (group by order date).

data.groupBy("order_date").agg(sum("total_price").alias("daily_sales")).orderBy("order_date").show()

+----------+------------------+
|order_date|       daily_sales|
+----------+------------------+
|2025-04-04| 7021.080015182495|
|2025-04-05| 8480.389987945557|
|2025-04-06| 7547.710021972656|
|2025-04-07|3494.9200143814087|
|2025-04-08| 3553.730007171631|
|2025-04-09|  5717.35004234314|
|2025-04-10| 5211.999990463257|
|2025-04-11| 4617.830003738403|
|2025-04-12|4297.3699951171875|
|2025-04-13|3384.1500091552734|
|2025-04-14| 5497.669965744019|
|2025-04-15| 4943.779993057251|
|2025-04-16| 6910.949971199036|
|2025-04-17| 3809.460005760193|
|2025-04-18| 6823.180011749268|
|2025-04-19|7357.6599798202515|
|2025-04-20| 4474.310013771057|
|2025-04-21| 7550.469969749451|
|2025-04-22| 6307.819986343384|
|2025-04-23| 5258.940028190613|
+----------+------------------+
only showing top 20 rows


In [21]:
#Find the return rate = Returned Orders / Total Orders per category.

total_orders = data.groupBy("product_category").count()

returned_orders = data.filter(col("order_status") == "Returned").groupBy("product_category").count().withColumnRenamed("count", "returned_count")

return_rate = total_orders.join(returned_orders, "product_category", "left").fillna(0).withColumn("return_rate", col("returned_count") / col("count"))

return_rate.show()

+----------------+-----+--------------+-------------------+
|product_category|count|returned_count|        return_rate|
+----------------+-----+--------------+-------------------+
|         Kitchen|  203|            55|  0.270935960591133|
|         Fashion|  188|            47|               0.25|
|     Electronics|  223|            45|0.20179372197309417|
|           Books|  193|            48|0.24870466321243523|
|      Home Decor|  193|            47|0.24352331606217617|
+----------------+-----+--------------+-------------------+



In [22]:
#Detect high-value customers (users who spent more than 5000 total).

data.groupBy("user_id").agg(sum("total_price").alias("total_spent")).filter(col("total_spent") > 5000).show()

+-------+-----------+
|user_id|total_spent|
+-------+-----------+
+-------+-----------+

